# บทเรียนที่ 18 (ต่อเนื่อง): ใบเสร็จที่พิสูจน์ว่า *มนุษย์* เป็นผู้อนุญาตการกระทำ

บทเรียนนี้พิสูจน์สิ่งที่ **ตัวแทน** ทำและสิ่งที่ **ประตู** ตัดสิน โน้ตบุ๊คนี้เพิ่มครึ่งที่ขาดหายไป: พิสูจน์ว่ามี **มนุษย์ที่มีชื่อเสียงรับรอง** การกระทำ **ที่แน่นอน** — ลายเซ็นแยกที่มนุษย์ถืออยู่เหนือการกระทำตามแบบฉบับเต็มรูปแบบ, ตรวจสอบแบบออฟไลน์

วัตถุสองอย่างที่นี่ใช้ **รูปร่างซองเดียวกับใบเสร็จของบทเรียน**: payload แบบแบนที่มีฟิลด์ `type` ลงลายมือชื่อโดย Ed25519 เหนือ SHA-256 ของไบต์ payload แบบฉบับ, มีอ็อบเจกต์ `signature` ที่มีโครงสร้างแนบอยู่ (และไม่รวมอยู่ในไบต์ที่ลงลายมือชื่อ) ใบเสร็จอนุมัติเป็น `type` ใหม่ (`human.approval.v1`) เคียงข้างประเภทของการกระทำ ดังนั้น `verify_chain` หนึ่งคำสั่งจึงครอบคลุมวัตถุทั้งสองชนิดด้วยเส้นทางโค้ดเดียวกับที่คุณสร้างในโน้ตบุ๊คหลัก นี่คือรูปร่างของเส้นทางร่วมลายเซ็นใน Internet-Draft ที่บทเรียนนี้ปฏิบัติตาม (draft-farley-acta-signed-receipts)

การอัปเกรดที่ตั้งใจหนึ่งอย่างเหนือผู้ตรวจสอบจำลองในโน้ตบุ๊คหลัก: ผู้ตรวจสอบที่นี่แก้ไข `signature.key_id` กับ **ทะเบียนกุญแจที่ตรึงไว้** แทนที่จะไว้วางใจคีย์สาธารณะที่บรรทุกอยู่ในใบเสร็จ พฤติกรรมนั้นเป็นท่าทีการผลิตที่เช็กลิสต์ของบทเรียนเองแนะนำ ("เผยแพร่คีย์ตรวจสอบสาธารณะ") และนั่นคือสิ่งที่ทำให้การปลอมแปลงเป็นการปฏิเสธแทนการเลี่ยงผ่านการนำกุญแจของตัวเองมาใช้

กฎที่โน้ตบุ๊คนี้สอน: **การอนุมัติที่ลงลายมือชื่อไม่ใช่อำนาจโดยตัวมันเอง** อำนาจมีอยู่ก็ต่อเมื่่อใบเสร็จอนุมัติและใบเสร็จการกระทำยังผูกติดอยู่กับการกระทำแบบฉบับเดียวกันในเวลาที่ทำการ, ภายใต้นโยบาย, คีย์, และวันหมดอายุที่ยังคงใช้งานได้, และการอนุมัติที่ยังไม่ถูกใช้ ทุกความล้มเหลวจะปฏิเสธด้วย **เหตุผลชัดเจนต่างกัน** ดังนั้นคุณสามารถบอกได้ว่า *อำนาจล้าสมัยแล้ว* แตกต่างจาก *การกระทำที่ดำเนินการเปลี่ยนแปลงไป*


In [1]:
# These are already the Lesson 18 dependencies — no new packages.
# %pip install pynacl jcs
import base64, copy, hashlib
from jcs import canonicalize                      # RFC 8785 canonical JSON
from nacl.signing import SigningKey, VerifyKey
# CryptoError is the common base of BadSignatureError AND the ValueError pynacl
# raises for a wrong-length signature — catch the base so verification fails
# closed on ANY bad signature, not just the forged-but-correct-length one.
from nacl.exceptions import CryptoError

# Same helpers as the main notebook.
def b64url_nopad(data: bytes) -> str:
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    return base64.urlsafe_b64decode(s + "=" * ((4 - len(s) % 4) % 4))

def sha256_canonical(obj) -> str:
    """SHA-256 of an object's JCS-canonical JSON form (same helper as the lesson)."""
    return f"sha256:{hashlib.sha256(canonicalize(obj)).hexdigest()}"

## การกระทำที่แน่นอน

หน่วยของการอนุมัติคือ **วัตถุการกระทำแบบมาตรฐาน** — ไม่ใช่ป้ายชื่อที่คลุมเครือเช่น "อนุมัติการคืนเงิน" แต่เป็นการกระทำที่ระบุอย่างแม่นยำและครบถ้วน การเซ็นชื่อบนวัตถุทั้งหมด (และสกัดรหัสย่อจากมัน) คือสิ่งที่ทำให้เราสามารถพิสูจน์ได้ในภายหลังว่ามนุษย์อนุมัติ *สิ่งนี้* เท่านั้นและไม่มีอย่างอื่น


In [2]:
action = {
    "action_type": "refund.issue",
    "params": {"order_id": "A-1029", "amount_usd": 4200, "to": "acct_88"},
    "policy_id": "refunds-v3",
}
print("action digest:", sha256_canonical(action))

action digest: sha256:fba342ad8447b491a089d7a09d4ac58f1a835c504e58f8d832db04f65bb62a25


## ซองเดียว สองอำนาจ

ใบเสร็จทุกใบคือซองบทเรียน: ข้อมูลเรียบง่ายที่มีฟิลด์ `type` และวัตถุ `signature` (`alg`, `sig`, `key_id`) ซึ่ง **ไม่ใช่** ส่วนหนึ่งของไบต์ที่ลงนามแล้ว `verify_envelope` คือการตรวจสอบโครงสร้างและลายเซ็นร่วมกันสำหรับใบเสร็จทั้งสองประเภท; ซึ่ง **ทะเบียนกุญแจที่กำหนด** ที่ใช้แก้ไข `signature.key_id` คือสิ่งที่แยกอำนาจออกจากกัน:

- **ใบเสร็จอนุมัติ** (`human.approval.v1`) — ผู้อนุมัติที่มีชื่อ, การกระทำแบบ canonical เต็มรูปแบบ **และ digest ของมัน**, `policy_version`, เวลาที่ออกและหมดอายุ การใช้ครั้งเดียวถูกติดตามในระดับ chain.
- **ใบเสร็จการกระทำ** (`agent.action.v1`) — ตัวตนของเอเจนต์, `run_id`, digest ของการกระทำ canonical เดียวกัน, ผลลัพธ์การดำเนินการ + เวลาที่บันทึก, และ `parent_approval_ref`: คือ `receipt_hash` ของการอนุมัติ, ตามรูปแบบเดียวกับ `previous_receipt_hash` ใน chain ของบทเรียน.

ฟิลด์ `action_digest` ที่ใช้ร่วมกันคือการเชื่อมโยงที่ binding พึ่งพา `key_id` อยู่ในวัตถุลายเซ็นเป็นเพียงตัวช่วยค้นหาเท่านั้น: การชี้ไปยังกุญแจ pinned ที่ต่างกันจะทำให้การตรวจสอบลายเซ็นล้มเหลว จึงไม่มีผลอะไร.


In [3]:
# ---- pinned key registries: SEPARATE authorities, one envelope shape ----------
# Published out of band (the lesson checklist's JWK-Set pattern); the verifier
# NEVER trusts a key carried inside a receipt.
approver_sk = SigningKey.generate()
agent_sk    = SigningKey.generate()
APPROVER_KEYS = {"approver-key-1": b64url_nopad(bytes(approver_sk.verify_key))}
AGENT_KEYS    = {"agent-key-1":    b64url_nopad(bytes(agent_sk.verify_key))}

# The policy the approval is granted under. If this moves after approval, the
# approval is STALE even though its signature still verifies.
CURRENT_POLICY = {"policy_version": "refunds-v3"}

def sign_receipt(payload: dict, sk: SigningKey, key_id: str) -> dict:
    """Same signing pipeline as the lesson: Ed25519 over the SHA-256 of the
    canonical payload; the signature object is NOT part of the signed bytes."""
    message_hash = hashlib.sha256(canonicalize(payload)).digest()
    return {
        **payload,
        "signature": {"alg": "EdDSA", "sig": b64url_nopad(sk.sign(message_hash).signature), "key_id": key_id},
    }

def verify_envelope(receipt, expected_type: str, trusted_keys: dict):
    """The SHARED verifier contract for any receipt kind; the caller picks which
    pinned registry (authority) resolves key_id. Fails closed on ANY
    attacker-shaped input: malformed is a refusal, never a crash."""
    if not isinstance(receipt, dict) or not isinstance(receipt.get("signature"), dict):
        return (False, "receipt malformed (not an object with a signature object)")
    sig_obj = receipt["signature"]
    if sig_obj.get("alg") != "EdDSA":
        return (False, "unsupported signature alg")
    if receipt.get("type") != expected_type:
        return (False, f"wrong receipt type (expected {expected_type})")
    # Key freshness is part of authority: a key_id rotated out of the pinned
    # registry confers nothing, even with a valid signature.
    pub = trusted_keys.get(sig_obj.get("key_id"))
    if pub is None:
        return (False, f"stale authority: key_id {sig_obj.get('key_id')!r} is not in the pinned registry (unknown or rotated out)")
    # Reconstruct the signed bytes exactly as the lesson does: everything except
    # the signature object, canonicalized, hashed.
    payload = {k: v for k, v in receipt.items() if k != "signature"}
    try:
        message_hash = hashlib.sha256(canonicalize(payload)).digest()
        VerifyKey(b64url_decode(pub)).verify(message_hash, b64url_decode(sig_obj.get("sig") or ""))
    except (CryptoError, TypeError, ValueError, base64.binascii.Error):
        return (False, "signature invalid (forged, tampered, or malformed)")
    return (True, "envelope ok")

def human_approval(action, approver_id, approved_at, sk=approver_sk,
                   key_id="approver-key-1", policy_version=None, expires_at=None):
    # deepcopy: the receipt must be an immutable record of what was approved —
    # a live reference would let a later mutation of `action` silently change the
    # signed payload. Digest the SNAPSHOT so the two can never diverge.
    approved_action = copy.deepcopy(action)
    payload = {
        "type": "human.approval.v1",
        "approver_id": approver_id,
        "action": approved_action,                       # the FULL canonical action
        "action_digest": sha256_canonical(approved_action),  # the join field
        "policy_version": policy_version or CURRENT_POLICY["policy_version"],
        "approved_at": approved_at,                      # ISO-8601 Zulu, like the lesson
        "expires_at": expires_at or approved_at[:11] + "23:59:59Z",
    }
    return sign_receipt(payload, sk, key_id)

In [4]:
approval = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T15:04:05Z",
                          expires_at="2026-07-08T15:19:05Z")
print(verify_envelope(approval, "human.approval.v1", APPROVER_KEYS))
print("binds digest:", approval["action_digest"][:23], "…  under", approval["policy_version"])

(True, 'envelope ok')
binds digest: sha256:fba342ad8447b491 …  under refunds-v3


## `verify_chain`: จุดที่กำหนดการผูกมัดจริง ๆ

`verify_chain` ไม่ใช่แค่ตัวห่อสะดวกของการตรวจสอบลายเซ็นสองครั้ง แต่มันคือจุดเดียวที่ตรวจสอบร่วมกันระหว่าง `action_digest` แบบมาตรฐานร่วมกัน, นโยบาย/กุญแจ/การหมดอายุของการอนุมัติที่ **ใหม่สด**, และการ **ใช้งานครั้งเดียว** ของการอนุมัติ เทียบกับการกระทำที่กำลังดำเนินการ *ในตอนนี้*

ทุกความล้มเหลวจะปฏิเสธด้วย **เหตุผลที่แตกต่างกัน** เพื่อให้ผู้อ่านที่ได้รับการปฏิเสธสามารถบอกได้ว่าอำนาจสิทธิ์ล้าสมัยไหม (นโยบายเปลี่ยน, กุญแจหมุนเวียน, การอนุมัติหมดอายุ, การอนุมัติถูกใช้แล้ว) หรือการกระทำที่ดำเนินการเปลี่ยนจากการอนุมัติที่ยังใช้ได้อยู่ (การแทนที่ digest)


In [5]:
def receipt_hash(receipt: dict) -> str:
    """Content-derived id of a COMPLETE receipt (including its signature) —
    the same convention as previous_receipt_hash in the lesson's chain."""
    return sha256_canonical(receipt)

def agent_receipt(action, approval, executed_at, sk=agent_sk, key_id="agent-key-1"):
    executed_action = copy.deepcopy(action)    # snapshot, same reason as the approval
    payload = {
        "type": "agent.action.v1",
        "agent_id": "agent:refunds-bot",
        "run_id": "run-0001",
        "action": executed_action,
        "action_digest": sha256_canonical(executed_action),  # same join field
        "parent_approval_ref": receipt_hash(approval),
        "outcome": "performed",
        "executed_at": executed_at,
    }
    return sign_receipt(payload, sk, key_id)

_consumed = set()

def verify_chain(action_being_executed, approval, agent_rcpt, now: str):
    """One code path covers both receipt kinds (same envelope), then checks the
    things that only make sense TOGETHER: shared digest, freshness, consumption.
    `now` is an ISO-8601 Zulu timestamp; Zulu strings compare correctly as strings."""
    # 1. Shared envelope contract, separate authorities.
    ok, why = verify_envelope(approval, "human.approval.v1", APPROVER_KEYS)
    if not ok: return (False, f"approval: {why}")
    ok, why = verify_envelope(agent_rcpt, "agent.action.v1", AGENT_KEYS)
    if not ok: return (False, f"agent receipt: {why}")

    # 2. The join: BOTH receipts must bind the digest of the action being executed
    #    right now. A valid approval for a DIFFERENT action is substitution, and it
    #    gets its own reason — this is "the executed action changed".
    executing_digest = sha256_canonical(action_being_executed)
    if approval.get("action_digest") != executing_digest or approval.get("action") != action_being_executed:
        return (False, "digest substitution: the approval binds a different canonical action than the one being executed")
    if agent_rcpt.get("action_digest") != executing_digest or agent_rcpt.get("action") != action_being_executed:
        return (False, "digest substitution: the agent receipt binds a different canonical action than the one being executed")
    if agent_rcpt.get("parent_approval_ref") != receipt_hash(approval):
        return (False, "agent receipt is not bound to this approval")

    # 3. Freshness: a valid signature over stale authority is still a refusal —
    #    each staleness gets its own reason, distinct from substitution above.
    if approval.get("policy_version") != CURRENT_POLICY["policy_version"]:
        return (False, f"stale authority: approved under policy {approval.get('policy_version')!r}, current is {CURRENT_POLICY['policy_version']!r}")
    expires = approval.get("expires_at")
    if not isinstance(expires, str) or not expires or now >= expires:
        return (False, "stale authority: approval expired before execution")

    # 4. One-time consumption: an approval authorizes ONE execution.
    ref = receipt_hash(approval)
    if ref in _consumed:
        return (False, "approval already consumed (replay refused)")
    _consumed.add(ref)
    return (True, f"approved by {approval['approver_id']}, executed by {agent_rcpt['agent_id']}")

def execute(action, approval, agent_rcpt, now):
    ok, why = verify_chain(action, approval, agent_rcpt, now)
    return (ok, "executed" if ok else why)

receipt = agent_receipt(action, approval, "2026-07-08T15:04:06Z")
print(execute(action, approval, receipt, now="2026-07-08T15:04:07Z"))

(True, 'executed')


## สิ่งที่การเชื่อมโยงจับได้

กรณีทั้งหมดต่อไปนี้ล้มเหลว **แบบปิด** ด้วย **เหตุผลที่แตกต่างกัน** บล็อกแรกเป็นชุดคลาสสิก (การปลอมแปลง, ตัวแทนสับสน, การเล่นซ้ำ, การปลอมแปลงทั้งสองฝ่ายอำนาจ, ข้อมูลนำเข้าที่ผิดรูปแบบ) บล็อกที่สองคือคู่ที่ทำให้คุณสมบัตินั้นเป็นจริงแทนที่จะเป็นการกล่าวอ้าง:

- **อำนาจที่ล้าสมัย** — ลายเซ็นยังถูกต้องอยู่ แต่เวอร์ชันนโยบายเปลี่ยนไป กุญแจผู้อนุมัติถูกหมุนออกจากทะเบียนที่ตรึงไว้ หรือการอนุมัติหมดอายุไปก่อนการดำเนินการ;
- **การแทนที่รหัสย่อ** — ใบเสร็จการดำเนินการที่ถูกลงนามอย่างถูกต้องซึ่ง `parent_approval_ref` ชี้ไปยังการอนุมัติ *ที่แท้จริง* แต่รหัสย่อของการดำเนินการที่เป็นมาตรฐานของการอนุมัตินั้นไม่ตรงกับการดำเนินการที่กระทำจริง


In [6]:
NOW = "2026-07-08T15:05:00Z"

# 1. tamper: change the amount after approval — the executed action changed.
tampered = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("tamper              ->", verify_chain(tampered, approval, agent_receipt(tampered, approval, NOW), NOW))

# 2. confused deputy: valid approval for action A, presented to execute action B.
action_b = {**action, "action_type": "wire.send"}
print("confused-deputy     ->", verify_chain(action_b, approval, agent_receipt(action_b, approval, NOW), NOW))

# 3. replay: the approval was consumed by the successful execution above.
print("replay              ->", execute(action, approval, agent_receipt(action, approval, NOW), NOW))

# 4. forged approval: attacker signs with their own key but claims a pinned key_id.
mallory_sk = SigningKey.generate()
forged = human_approval(action, "mallory", NOW, sk=mallory_sk)
print("forged-approval     ->", verify_chain(action, forged, agent_receipt(action, forged, NOW), NOW))

# A fresh, un-consumed approval so the agent-side cases fail on their OWN check.
fresh = human_approval(action, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")

# 5. self-minted agent receipt: attacker's own agent key, refused by the pinned registry.
mallory_agent = agent_receipt(action, fresh, NOW, sk=SigningKey.generate())
print("self-minted-agent   ->", verify_chain(action, fresh, mallory_agent, NOW))

# 6. wrong-action agent receipt: real agent key, but the receipt binds a different action.
wrong_action = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("wrong-action-agent  ->", verify_chain(action, fresh, agent_receipt(wrong_action, fresh, NOW), NOW))

# 7. malformed input: structurally broken receipts refuse cleanly, they never crash.
print("malformed-approval  ->", verify_chain(action, {"type": "human.approval.v1"}, agent_receipt(action, fresh, NOW), NOW))
print("malformed-agent     ->", verify_chain(action, fresh, {"nope": "not a receipt"}, NOW))

# 8. wrong-length signature: valid base64, not 64 bytes — refused, not crashed.
badlen = {**fresh, "signature": {**fresh["signature"], "sig": "AAAA"}}
print("wrong-len-sig       ->", verify_chain(action, badlen, agent_receipt(action, fresh, NOW), NOW))

# 9. non-object receipt: a list refuses cleanly instead of raising AttributeError.
print("nonobject-receipt   ->", verify_chain(action, [1, 2], agent_receipt(action, fresh, NOW), NOW))

print()
print("--- the two negative controls that make the property real ---")

# 10. STALE POLICY: signature still valid, but policy moved between approval and
#     execution. Authority is decided at execution time, not signing time.
CURRENT_POLICY["policy_version"] = "refunds-v4"
print("stale-policy        ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
CURRENT_POLICY["policy_version"] = "refunds-v3"   # restore for the cases below

# 11. STALE KEY: the approver key is rotated out of the pinned registry after
#     signing. The signature bytes still verify against the old key — but the old
#     key no longer confers authority.
rotated_out = APPROVER_KEYS.pop("approver-key-1")
print("stale-key           ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
APPROVER_KEYS["approver-key-1"] = rotated_out     # restore

# 12. EXPIRED: approval was valid when signed, but execution came too late.
expired = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T14:00:00Z",
                         expires_at="2026-07-08T14:01:00Z")
print("expired-approval    ->", verify_chain(action, expired, agent_receipt(action, expired, NOW), NOW))

# 13. DIGEST SUBSTITUTION: a validly signed agent receipt whose parent_approval_ref
#     points at a REAL approval — but that approval binds action B, and the agent
#     is executing action A. Distinct reason from every staleness above.
approval_b = human_approval(action_b, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")
substituted = agent_receipt(action, approval_b, NOW)   # executing `action`, ref -> approval of action_b
print("digest-substitution ->", verify_chain(action, approval_b, substituted, NOW))

tamper              -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
confused-deputy     -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
replay              -> (False, 'approval already consumed (replay refused)')
forged-approval     -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
self-minted-agent   -> (False, 'agent receipt: signature invalid (forged, tampered, or malformed)')
wrong-action-agent  -> (False, 'digest substitution: the agent receipt binds a different canonical action than the one being executed')
malformed-approval  -> (False, 'approval: receipt malformed (not an object with a signature object)')
malformed-agent     -> (False, 'agent receipt: receipt malformed (not an object with a signature object)')
wrong-len-sig       -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
nonobject-receipt   -> (Fa

## สิ่งที่สิ่งนี้พิสูจน์ — และสิ่งที่มันไม่พิสูจน์

**พิสูจน์:** มนุษย์ที่มีชื่อได้รับรอง *การดำเนินการมาตรฐานเฉพาะนี้* (การดำเนินการทั้งหมด + ดิจิสต์, ลงลายมือชื่อด้วยกุญแจที่ตรวจสอบได้จากรีจิสตรีที่ตรึงไว้) และตัวแทนได้ดำเนินการ *ตามการดำเนินการที่ได้รับการอนุมัตินั้นอย่างแม่นยำ* (ดิจิสต์เดียวกัน, ใบเสร็จที่ผูกไว้กับการอนุมัติด้วย `receipt_hash` ซึ่งเป็นคอนเวนชันของสายของบทเรียนเอง) — ในขณะที่เวอร์ชันนโยบายของการอนุมัติ, กุญแจ, และวันหมดอายุยังคงเป็นปัจจุบัน, เพียงครั้งเดียว หากฝั่งใดฝั่งหนึ่งเปลี่ยนแปลง สายจะล้มเหลวอย่างปิด และเหตุผลการปฏิเสธจะบอกคุณว่า **คุณสมบัติใด** แตกหัก: อำนาจหมดอายุเทียบกับการดำเนินการที่เปลี่ยนแปลง

**ไม่ได้พิสูจน์:** ว่า UI การอนุมัติแสดงสิ่งที่มนุษย์คิดว่ากำลังลงนาม (WYSIWYS เป็นปัญหาเฉพาะตัวของมันเอง), ว่ากุญแจไม่ได้ถูกบังคับหรือขโมยก่อนหมุนเวียน, หรือว่าผลกระทบจากลำดับต่อไปตรงกับการดำเนินการ การลงนาม ≠ การได้รับอนุญาต: ลายเซ็นที่ถูกต้องที่อยู่บนมาตรการหมดอายุ, กุญแจหมุนเวียน, หน้าต่างหมดอายุ, หรือดิจิสต์ที่ต่างไปไม่ได้ให้สิ่งใดที่นี่

ใบเสร็จสองแบบแชร์ซองของบทเรียนและเส้นทางโค้ด `verify_chain` ทางเดียวโดยเจตนา: การผูกที่คุณสร้างสำหรับใบเสร็จการดำเนินการในสมุดบันทึกหลักคือโค้ดเดียวกันที่ตรวจสอบการอนุมัติของมนุษย์ สัญญาตรวจสอบหนึ่งฉบับ, อำนาจตรึงแยกต่างหาก, รวมกันด้วยดิจิสต์ของการดำเนินการมาตรฐานและไม่มีอะไรอื่น


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ปฏิเสธความรับผิดชอบ**:
เอกสารนี้ได้รับการแปลโดยใช้บริการแปลภาษา AI [Co-op Translator](https://github.com/Azure/co-op-translator) ขณะที่เราพยายามให้ความถูกต้อง โปรดทราบว่าการแปลโดยอัตโนมัติอาจมีข้อผิดพลาดหรือความไม่ถูกต้อง เอกสารต้นฉบับในภาษาต้นทางควรถูกพิจารณาเป็นแหล่งข้อมูลที่เชื่อถือได้ สำหรับข้อมูลที่สำคัญ แนะนำให้ใช้การแปลโดยมนุษย์มืออาชีพ เราไม่รับผิดชอบต่อความเข้าใจผิดหรือการตีความที่ผิดพลาดที่เกิดขึ้นจากการใช้การแปลนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
